# E-Commerce Sales Data Preparation

This notebook cleans the source workbook, derives sales metrics, exports a cleaned CSV, and optionally loads the result into MySQL.

**Input required:** `Ecommerce_Unclean_Project.xlsx` (not included in the repository). Run cells in order. Review the metric caveats in `docs/data-quality-notes.md`.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

## 1. Load the source workbook

The notebook looks in `data/` when run from the repository root, or in `../data/` when run from the `notebooks/` directory.

In [ ]:
candidate_paths = [
    Path("data/Ecommerce_Unclean_Project.xlsx"),
    Path("../data/Ecommerce_Unclean_Project.xlsx"),
    Path("Ecommerce_Unclean_Project.xlsx"),
]
input_path = next((path for path in candidate_paths if path.exists()), None)
if input_path is None:
    raise FileNotFoundError(
        "Place the licensed source workbook at data/Ecommerce_Unclean_Project.xlsx "
        "or update candidate_paths to its local location."
    )

df = pd.read_excel(input_path)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
df.head()

## 2. Inspect and clean the data

In [ ]:
display(df.info())
display(df.describe(include="all").transpose())
print("Missing values before cleaning:")
display(df.isna().sum().sort_values(ascending=False))
print(f"Duplicate rows before cleaning: {df.duplicated().sum():,}")

In [ ]:
# Remove exact duplicate rows.
df = df.drop_duplicates().copy()

# Normalize common text placeholders to missing values, then trim text columns.
df = df.replace(["N/A", "NULL", "null", ""], np.nan)
for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].str.strip()
    df[column] = df[column].replace("", np.nan)

# Standardize selected labels if those columns exist.
for column in ["Customer_Name", "City", "State"]:
    if column in df.columns:
        df[column] = df[column].str.title()

# Lightweight quality filter only; containing '@' does not fully validate an email.
if "Email" in df.columns:
    df = df[df["Email"].astype("string").str.contains("@", na=False)].copy()

# Parse dates and numeric fields safely.
for column in ["Order_Date", "Delivery_Date"]:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

for column in ["Qty", "Unit_Price", "Discount"]:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

# Keep rows with a positive quantity.
if "Qty" in df.columns:
    df = df[df["Qty"] > 0].copy()

# Apply the assumptions used in the supplied version of the project.
if "Discount" in df.columns:
    df["Discount"] = df["Discount"].fillna(0)
if "Phone" in df.columns:
    df["Phone"] = df["Phone"].fillna("Unknown")
if {"Delivery_Date", "Order_Date"}.issubset(df.columns):
    # Note: this assumes missing delivery dates equal the order date; see docs/data-quality-notes.md.
    df["Delivery_Date"] = df["Delivery_Date"].fillna(df["Order_Date"])

print(f"Rows remaining after cleaning: {len(df):,}")
display(df.head())

## 3. Feature engineering

`Profit` is currently estimated as 20% of `Net_Amount`, matching the submitted project logic. It is not actual accounting profit unless the margin assumption is validated.

In [ ]:
required_columns = {"Qty", "Unit_Price", "Discount", "Order_Date"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise KeyError(f"Required columns missing from source workbook: {sorted(missing_columns)}")

df["Sales"] = df["Qty"] * df["Unit_Price"]
df["Net_Amount"] = df["Sales"] - (df["Sales"] * df["Discount"] / 100)
df["Profit"] = df["Net_Amount"] * 0.20
df["Month"] = df["Order_Date"].dt.month_name()
df["Year"] = df["Order_Date"].dt.year
df["Weekday"] = df["Order_Date"].dt.day_name()

display(df[["Qty", "Unit_Price", "Discount", "Sales", "Net_Amount", "Profit", "Month", "Year", "Weekday"]].head())

## 4. Validate and export

In [ ]:
print(f"Shape: {df.shape}")
print(f"Duplicate rows: {df.duplicated().sum():,}")
print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False))
display(df.describe(include="all").transpose())

output_path = input_path.parent / "Clean_Ecommerce.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Cleaned data exported to: {output_path.resolve()}")

## 5. Optional: load the cleaned data into MySQL

Set database credentials through environment variables before running this section. Do not hard-code passwords or commit them to source control.

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

DB_USER = os.getenv("ECOMMERCE_DB_USER", "root")
DB_PASSWORD = os.getenv("ECOMMERCE_DB_PASSWORD")
DB_HOST = os.getenv("ECOMMERCE_DB_HOST", "localhost")
DB_PORT = int(os.getenv("ECOMMERCE_DB_PORT", "3306"))
DB_NAME = os.getenv("ECOMMERCE_DB_NAME", "ecommerce")

if not DB_PASSWORD:
    raise ValueError("Set ECOMMERCE_DB_PASSWORD before connecting to MySQL.")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(connection_url, pool_pre_ping=True)

# `replace` recreates the table, so use this only when overwriting the local table is intended.
df.to_sql(name="orders", con=engine, if_exists="replace", index=False)
print("Loaded cleaned records into MySQL table: ecommerce.orders")

## Next step

Run `sql/ecommerce_sales_analysis.sql` in MySQL Workbench after the table is loaded. Open `dashboard/ecommerce_sales_dashboard.pbix` in Power BI Desktop and confirm that its data source and measures refresh correctly.